# MolMerger Solubility Prediction Automation

This notebook predicts aqueous LogS for every CSV file in an input folder. Each input CSV must contain a `SMILES` column. Results are written file-by-file to the output folder, and large files are processed in chunks to reduce memory usage.

## 0.Verification of the environment

In [1]:
import torch
import torchdata
import torchdata.datapipes.iter
import dgl

print(torch.__version__)
print(torchdata.__version__)
print(dgl.__version__)

/home/cenking/miniconda3/envs/MolmergerSolubilityPrediction/lib/python3.9/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


2.2.1+cu121
0.7.1
2.1.0


## 1. Absolute Paths and Settings

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path

PROJECT_DIR = Path(r"/home/cenking/VsCode/MolmergerSolubilityPrediction")
INPUT_DIR = Path(r"/home/cenking/VsCode/Data_Port/Candidate")
OUTPUT_DIR = Path(r"/home/cenking/VsCode/Data_Port/MolmergerSolubilityPrediction")

RUN_DIR = PROJECT_DIR / "runs" / "molmerger_train"
CHECKPOINT_DIR = RUN_DIR / "checkpoints" / "best_model"
CONFIG_PATH = RUN_DIR / "training_config.json"

SMILES_COLUMN = "SMILES"
SOLVENT_SMILES = "CS(C)=O"
CHUNK_SIZE = 1024
SHARD_SIZE = 8192
OVERWRITE_OUTPUTS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project: {PROJECT_DIR}")
print(f"Input:   {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Model:   {CHECKPOINT_DIR}")

Project: /home/cenking/VsCode/MolmergerSolubilityPrediction
Input:   /home/cenking/VsCode/Data_Port/Candidate
Output:  /home/cenking/VsCode/Data_Port/MolmergerSolubilityPrediction
Model:   /home/cenking/VsCode/MolmergerSolubilityPrediction/runs/molmerger_train/checkpoints/best_model


## 2. Imports and Model Loading

In [4]:
import json
import sys
import tempfile
import warnings

import scipy.stats as st
if not hasattr(st, "gilbrat") and hasattr(st, "gibrat"):
    st.gilbrat = st.gibrat

import deepchem as dc
import numpy as np
import pandas as pd
from rdkit import Chem
from tqdm.notebook import tqdm
import torch

_original_torch_load = torch.load

def torch_load_cpu(*args, **kwargs):
    kwargs.setdefault("map_location", torch.device("cpu"))
    return _original_torch_load(*args, **kwargs)

torch.load = torch_load_cpu

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from attentivefp_model import AttentiveFPModel
from molmerger_utils import MolMerger, MolMergerFeaturizer

warnings.filterwarnings("ignore", category=UserWarning)

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input folder not found: {INPUT_DIR}")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Training config not found: {CONFIG_PATH}")
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {CHECKPOINT_DIR}")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

model = AttentiveFPModel(
    n_tasks=1,
    mode="regression",
    num_layers=config.get("num_layers", 3),
    num_timesteps=config.get("num_timesteps", 3),
    graph_feat_size=config.get("graph_feat_size", 200),
    dropout=config.get("dropout", 0.2),
    batch_size=config.get("batch_size", 128),
    learning_rate=config.get("learning_rate", 0.001),
    model_dir=str(RUN_DIR / "automation_model"),
    use_gpu=False,
)
model.restore(model_dir=str(CHECKPOINT_DIR))
model.model.to("cpu")

featurizer = MolMergerFeaturizer(use_edges=True)
print("Model loaded.")
print("DeepChem model device:", model.device)

Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'pytorch_lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'


Model loaded.
DeepChem model device: cpu


## 3. Streaming Prediction Helpers

In [5]:
def output_path_for(input_csv: Path) -> Path:
    return OUTPUT_DIR / f"{input_csv.stem}_MolMerger_LogS.csv"


def merge_smiles_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for local_index, smiles in enumerate(chunk[SMILES_COLUMN].astype(str)):
        smiles = smiles.strip()
        row = {
            "_local_index": local_index,
            SMILES_COLUMN: smiles,
            "Smiles_Solvent": SOLVENT_SMILES,
            "Smiles_Merged": np.nan,
            "Predicted_LogS": np.nan,
            "Prediction_Status": "ok",
            "Prediction_Error": "",
        }
        try:
            row["Smiles_Merged"] = MolMerger(smiles, SOLVENT_SMILES)
        except Exception as exc:
            row["Prediction_Status"] = "failed"
            row["Prediction_Error"] = str(exc)
        rows.append(row)
    return pd.DataFrame(rows)


def predict_valid_rows(merged_df: pd.DataFrame) -> pd.DataFrame:
    valid_mask = merged_df["Prediction_Status"].eq("ok") & merged_df["Smiles_Merged"].notna()
    valid_indices = merged_df.index[valid_mask].tolist()
    if not valid_indices:
        return merged_df

    graph_indices = []
    graphs = []
    for row_index in valid_indices:
        merged_smiles = str(merged_df.at[row_index, "Smiles_Merged"])
        try:
            mol = Chem.MolFromSmiles(merged_smiles)
            if mol is None:
                raise ValueError("RDKit could not parse Smiles_Merged")
            graph = featurizer._featurize(mol)
        except Exception as exc:
            merged_df.at[row_index, "Prediction_Status"] = "failed"
            merged_df.at[row_index, "Prediction_Error"] = f"Featurization failed: {exc}"
            continue
        graph_indices.append(row_index)
        graphs.append(graph)

    if not graphs:
        return merged_df

    dataset = dc.data.NumpyDataset(
        X=np.asarray(graphs, dtype=object),
        y=np.zeros((len(graphs), 1), dtype=float),
        ids=np.asarray(graph_indices, dtype=object),
    )
    predictions = model.predict(dataset).reshape(-1)
    for row_index, prediction in zip(graph_indices, predictions):
        merged_df.at[row_index, "Predicted_LogS"] = float(prediction)

    return merged_df


def process_one_file(input_csv: Path) -> dict:
    output_csv = output_path_for(input_csv)
    if output_csv.exists() and not OVERWRITE_OUTPUTS:
        return {"file": input_csv.name, "status": "skipped_exists", "rows": 0, "failed": 0, "output": str(output_csv)}

    header = pd.read_csv(input_csv, nrows=0)
    if SMILES_COLUMN not in header.columns:
        return {"file": input_csv.name, "status": "skipped_missing_SMILES", "rows": 0, "failed": 0, "output": ""}

    if output_csv.exists():
        output_csv.unlink()

    total_rows = 0
    failed_rows = 0
    wrote_header = False

    chunks = pd.read_csv(input_csv, chunksize=CHUNK_SIZE)
    for chunk in tqdm(chunks, desc=f"Rows: {input_csv.name}", unit="chunk", leave=False):
        chunk = chunk.copy()
        merged_df = merge_smiles_chunk(chunk)
        merged_df = predict_valid_rows(merged_df)

        result_df = chunk.reset_index(drop=True).copy()
        result_df["Smiles_Solvent"] = merged_df["Smiles_Solvent"]
        result_df["Smiles_Merged"] = merged_df["Smiles_Merged"]
        result_df["Predicted_LogS"] = merged_df["Predicted_LogS"]
        result_df["Prediction_Status"] = merged_df["Prediction_Status"]
        result_df["Prediction_Error"] = merged_df["Prediction_Error"]

        result_df.to_csv(output_csv, mode="a", header=not wrote_header, index=False, encoding="utf-8-sig")
        wrote_header = True

        total_rows += len(result_df)
        failed_rows += int(result_df["Prediction_Status"].ne("ok").sum())

    return {"file": input_csv.name, "status": "done", "rows": total_rows, "failed": failed_rows, "output": str(output_csv)}

## 4. Run Folder Prediction

In [6]:
csv_files = sorted(INPUT_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).")
for path in csv_files:
    print(path.name)

summary_rows = []
for csv_file in tqdm(csv_files, desc="CSV files", unit="file"):
    summary_rows.append(process_one_file(csv_file))

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "prediction_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

display(summary_df)
print(f"Summary written to: {summary_path}")

Found 1 CSV file(s).
candidate_only_SMILES.csv


CSV files:   0%|          | 0/1 [00:00<?, ?file/s]

Rows: candidate_only_SMILES.csv: 0chunk [00:00, ?chunk/s]

,file,status,rows,failed,output
0,candidate_only_SMILES.csv,done,6605,354,/home/cenking/VsCode/Data_Port/MolmergerSolubi...


Summary written to: /home/cenking/VsCode/Data_Port/MolmergerSolubilityPrediction/prediction_summary.csv


## 5. Quick Output Check

In [7]:
output_files = sorted(OUTPUT_DIR.glob("*_MolMerger_LogS.csv"))
print(f"Generated {len(output_files)} output file(s).")
if output_files:
    preview_path = output_files[0]
    print(f"Preview: {preview_path}")
    display(pd.read_csv(preview_path).head())

Generated 2 output file(s).
Preview: /home/cenking/VsCode/Data_Port/MolmergerSolubilityPrediction/candidate_MolMerger_LogS.csv


,SMILES,Smiles_Solvent,Smiles_Merged,Predicted_LogS,Prediction_Status,Prediction_Error
0,CC[n+]1ccc(/C=C/C(C)=C/C=C/C(C)=C/C=C/C=C(C)/C...,O,CC[n+]1ccc(/C=C/C(C)=C/C=C/C(C)=C/C=C/C=C(C)/C...,-1.835442,ok,NaN
1,CN1CCSC1=CC=CC=CC=CC1=[N+](C)CCS1,O,C[N+]1=C2(~O~N3(C)CCSC3=CC=CC=CC=C2)SCC1,-2.464926,ok,NaN
2,CC=CC=CC=CC=CC=CC(=O)C1=C(O)[C@@H](CCC(=O)O)N(...,O,CC=CC=CC=CC=CC=CC(=O)C1=C2O~O~C(=O)(O)CC[C@H]2...,-3.334324,ok,NaN
3,CC1(C)C(C=CC2=C(Cl)C(=CC=C3N(CCCCS(=O)(=O)[O-]...,O,CC1(C)C(=CC=C2CCCC(C=CC3=[N+](CCCCS(=O)(=O)[O-...,-3.928039,ok,NaN
4,CCCCn1c(=CC=CC=CC=CC2=[N+](CCCC)c3cccc4cccc2c3...,O,CCCC[N+]1=C2C=CC=CC=CC=c3c4cccc5cccc(c54)n3(CC...,-7.558506,ok,NaN
